# TRM Compiler Pass Ordering — Real LLVM Training

Train a 60K-param TRM model on real LLVM optimization passes via CompilerGym, then benchmark.

**Problem:** `compiler_gym` depends on `gym<=0.21` which requires `distutils` (removed in Python 3.12).

**Solution:** Use Python 3.11 via a virtual environment. All heavy work runs in the venv via subprocess;
results are loaded back into the notebook for visualization.

**Runtime:** Change runtime type to T4 GPU for training.

In [9]:
# Cell 1: Clone repo
import os

REPO_URL = 'https://github.com/Cree0618/trm-youtubevids.git'
PROJECT_DIR = '/content/trm-youtubevids'

if not os.path.exists(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
else:
    !cd $PROJECT_DIR && git pull

os.chdir(PROJECT_DIR)
!pwd

Already up to date.
/content/trm-youtubevids


In [ ]:
# Cell 2: Install Python 3.11 and create venv (compiler_gym needs Python <= 3.11)
# gym<=0.21 uses distutils which was removed in Python 3.12

!sudo apt-get update -qq && sudo apt-get install -qq -y python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1
!python3.11 --version

VENV_DIR = '/content/trm-env'
if not os.path.exists(VENV_DIR):
    !python3.11 -m venv $VENV_DIR
    print(f'Created venv at {VENV_DIR}')
else:
    print(f'Venv already exists at {VENV_DIR}')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# Cell 3: Install all dependencies in the Python 3.11 venv
# This is the slow step (~5 min first time, cached after)

PIP = f'{VENV_DIR}/bin/pip'
PYTHON = f'{VENV_DIR}/bin/python'

!$PIP install --upgrade pip -q
!$PIP install torch numpy compiler_gym matplotlib -q

# Install our project package
!$PIP install -e $PROJECT_DIR -q

# Verify
!$PYTHON -c "import torch; print(f'PyTorch: {torch.__version__}')"
!$PYTHON -c "import compiler_gym; print(f'CompilerGym: {compiler_gym.__version__}')"
print('All dependencies installed in venv')

In [ ]:
# Cell 4: Quick smoke test — verify CompilerGym works with real LLVM

!$PYTHON -c "
import compiler_gym
env = compiler_gym.make('llvm-v0', benchmark='cbench-v1/qsort',
    observation_space='Autophase', reward_space='IrInstructionCountOz')
obs = env.reset()
print(f'Autophase features: {len(env.observation[\"Autophase\"])}')
print(f'Initial inst count: {env.observation[\"IrInstructionCount\"]}')
print(f'Action space: {env.action_space.n} actions')
# Try a few passes
for i in range(3):
    _, reward, done, _ = env.step(env.action_space.sample())
    inst = env.observation['IrInstructionCount']
    print(f'  Step {i}: inst={inst} reward={reward:.2f} done={done}')
env.close()
print('CompilerGym OK')
"

## Generate Training Traces (Real LLVM)

Runs real LLVM passes on 18 cbench benchmarks.
~30-60 min first time, cached after.

In [ ]:
# Cell 5: Generate traces with real CompilerGym

TRACES_PATH = f'{PROJECT_DIR}/trm_compiler_output/traces_real_llvm.json'

if os.path.exists(TRACES_PATH):
    print(f'Loading cached traces from {TRACES_PATH}')
    import json
    with open(TRACES_PATH) as f:
        traces_data = json.load(f)
    print(f'Loaded {len(traces_data)} traces')
else:
    print('Generating traces with real CompilerGym (this takes 30-60 min)...')
    !$PYTHON -c "
import time, json, sys
sys.path.insert(0, '$PROJECT_DIR')
from trm_compiler.data import generate_compiler_traces

t0 = time.time()
traces = generate_compiler_traces(
    benchmarks=['qsort','adpcm','blowfish','bzip2','dijkstra','sha',
                'gsm','ispell','jpeg-c','lame','patricia','rijndael',
                'stringsearch','susan','tiff2bw','tiff2rgba','tiffdither','tiffmedian'],
    episodes_per_benchmark=50,
    max_steps_per_episode=30,
    use_heuristic=False,
    output_path='$TRACES_PATH',
    seed=42,
)
print(f'Generated {len(traces)} traces in {time.time()-t0:.0f}s')
"

## Train TRM Model

Trains the 60K-param TRM model on real LLVM pass traces.
Runs in the Python 3.11 venv (CompilerGym compatibility).

In [ ]:
# Cell 6: Train the model

TRAIN_SCRIPT = f"{PROJECT_DIR}/trm_compiler/train_notebook.py"
MODEL_PATH = f'{PROJECT_DIR}/trm_compiler_output/trm_model_real.pt'
HISTORY_PATH = f'{PROJECT_DIR}/trm_compiler_output/training_history.json'

# Write training script
with open(TRAIN_SCRIPT, 'w') as f:
    f.write('''
import sys, time, json
sys.path.insert(0, sys.argv[1])

import torch
from torch.utils.data import DataLoader
from trm_compiler.data import load_traces, CompilerTraceDataset
from trm_compiler.model import TinyPassOrderingRefiner
from trm_compiler.training import train_one_epoch
from trm_compiler.env_wrapper import NUM_PASSES

config = {
    'traces_path': sys.argv[2],
    'model_path': sys.argv[3],
    'history_path': sys.argv[4],
    'epochs': int(sys.argv[5]),
    'batch_size': 64,
    'lr': 1e-3,
    'weight_decay': 1e-4,
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

traces = load_traces(config['traces_path'])
dataset = CompilerTraceDataset(traces)
dataloader = DataLoader(dataset, batch_size=config['batch_size'],
    shuffle=True, num_workers=2, drop_last=True,
    pin_memory=(device == 'cuda'))

model = TinyPassOrderingRefiner(
    observation_dim=56, latent_dim=64, hidden_dim=128,
    num_passes=NUM_PASSES, n_recursions=6,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} params, {len(dataset):,} records')

optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

history = {'epoch': [], 'total_loss': [], 'pass_loss': [], 'entropy': [], 'value_loss': []}
best_loss = float('inf')

for epoch in range(config['epochs']):
    t0 = time.time()
    losses = train_one_epoch(model, dataloader, optimizer, device)
    scheduler.step()
    history['epoch'].append(epoch + 1)
    for k in ['total_loss', 'pass_loss', 'value_loss']:
        history[k].append(losses[k])
    history['entropy'].append(losses.get('entropy', 0))
    elapsed = time.time() - t0
    print(f'Epoch {epoch+1:3d}/{config[\"epochs\"]} | '
          f'Loss: {losses[\"total_loss\"]:.4f} | Pass: {losses[\"pass_loss\"]:.4f} | '
          f'Entropy: {losses.get(\"entropy\", 0):.3f} | {elapsed:.1f}s')
    if losses['total_loss'] < best_loss:
        best_loss = losses['total_loss']
        torch.save(model.state_dict(), config['model_path'])

torch.save(model.state_dict(), config['model_path'].replace('.pt', '_final.pt'))
with open(config['history_path'], 'w') as f:
    json.dump(history, f)
print(f'Done. Best loss: {best_loss:.4f}')
''')

print(f'Written training script to {TRAIN_SCRIPT}')
print(f'Training for 30 epochs on real LLVM traces...')

!$PYTHON $TRAIN_SCRIPT $PROJECT_DIR $TRACES_PATH $MODEL_PATH $HISTORY_PATH 30

In [ ]:
# Cell 7: Plot training curves
import json
import matplotlib.pyplot as plt

with open(HISTORY_PATH) as f:
    history = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['epoch'], history['total_loss'], 'b-', lw=2)
ax1.set(xlabel='Epoch', ylabel='Loss', title='Total Loss', yscale='log')
ax1.grid(True, alpha=0.3)

ax2.plot(history['epoch'], history['pass_loss'], label='Pass Loss', lw=2)
ax2.plot(history['epoch'], history['entropy'], label='Entropy', lw=2)
ax2.plot(history['epoch'], history['value_loss'], label='Value Loss', lw=2)
ax2.set(xlabel='Epoch', ylabel='Loss', title='Components', yscale='log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Final: loss={history["total_loss"][-1]:.4f}, entropy={history["entropy"][-1]:.3f}')

## Benchmark — Real LLVM (CompilerGym)

Compares LLVM -Oz, Random(100), Greedy, and TRM on real LLVM passes.
Runs in the Python 3.11 venv.

In [ ]:
# Cell 8: Run full benchmark on real LLVM

BENCHMARK_SCRIPT = f"{PROJECT_DIR}/trm_compiler/benchmark_notebook.py"
RESULTS_PATH = f'{PROJECT_DIR}/trm_compiler_output/benchmark_results.json'

with open(BENCHMARK_SCRIPT, 'w') as f:
    f.write('''
import sys, json, time, math
sys.path.insert(0, sys.argv[1])

import torch
import numpy as np
import compiler_gym
from trm_compiler.model import TinyPassOrderingRefiner, rollout_closed_loop
from trm_compiler.env_wrapper import NUM_PASSES, pass_id_to_name, CompilerGymWrapper

device = 'cpu'  # inference doesn't need GPU
model_path = sys.argv[2]
results_path = sys.argv[3]
benchmarks = sys.argv[4].split(',')
max_steps = int(sys.argv[5])
seed = int(sys.argv[6])

# Load model
model = TinyPassOrderingRefiner(observation_dim=56, latent_dim=64, hidden_dim=128,
    num_passes=NUM_PASSES, n_recursions=6)
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.eval()

results = {}

for bench_id in benchmarks:
    full_id = f'cbench-v1/{bench_id}'
    print(f'\\nBenchmark: {bench_id}')
    print('-' * 60)
    bench_results = {}

    # 1. LLVM -Oz baseline
    try:
        env = compiler_gym.make('llvm-v0', benchmark=full_id,
            observation_space='Autophase', reward_space='IrInstructionCountOz')
        env.reset()
        oz_inst = env.observation['IrInstructionCountOz']
        base_inst = env.observation['IrInstructionCount']
        oz_reward = math.log(base_inst / max(oz_inst, 1))
        bench_results['llvm_Oz'] = {
            'total_reward': oz_reward,
            'initial_inst': base_inst,
            'final_inst': oz_inst,
            'reduction_pct': 1.0 - oz_inst / base_inst if base_inst > 0 else 0,
        }
        print(f'  LLVM Oz: reward={oz_reward:+.4f} inst={base_inst}->{oz_inst} ({(1-oz_inst/base_inst)*100:.1f}%)')
        env.close()
    except Exception as e:
        print(f'  LLVM Oz: failed ({e})')
        bench_results['llvm_Oz'] = {'total_reward': 0, 'error': str(e)}

    # 2. Random search (100 trials)
    rng = np.random.RandomState(seed)
    random_rewards = []
    best_random = float('-inf')
    for trial in range(100):
        try:
            env = compiler_gym.make('llvm-v0', benchmark=full_id,
                observation_space='Autophase', reward_space='IrInstructionCountOz')
            env.reset()
            total_r = 0
            for s in range(max_steps):
                action = rng.randint(0, env.action_space.n)
                _, r, done, _ = env.step(action)
                total_r += r
                if done: break
            random_rewards.append(total_r)
            best_random = max(best_random, total_r)
            env.close()
        except:
            pass
    if random_rewards:
        bench_results['random_100'] = {
            'total_reward': float(np.mean(random_rewards)),
            'best_reward': float(best_random),
            'std': float(np.std(random_rewards)),
        }
        print(f'  Random(100): mean={np.mean(random_rewards):+.4f} best={best_random:+.4f}')

    # 3. TRM closed-loop
    try:
        wrapper = CompilerGymWrapper(benchmark_id=full_id)
        wrapper.open()
        obs, initial_inst = wrapper.reset()

        from trm_compiler.model import rollout_closed_loop
        trace = rollout_closed_loop(model, wrapper, max_steps=max_steps, temperature=1.0, device=device)
        total_reward = trace[-1]['total_real_reward'] if trace else 0
        final_inst = wrapper.current_inst_count
        reduction = 1.0 - final_inst / initial_inst if initial_inst > 0 else 0

        bench_results['trm'] = {
            'total_reward': total_reward,
            'initial_inst': initial_inst,
            'final_inst': final_inst,
            'reduction_pct': reduction,
            'steps': len([s for s in trace if s['pass_id'] >= 0]),
            'sequence': [pass_id_to_name(s['pass_id']) for s in trace if s['pass_id'] >= 0],
        }
        print(f'  TRM: reward={total_reward:+.4f} inst={initial_inst}->{final_inst} ({reduction*100:.1f}%)')
        wrapper.close()
    except Exception as e:
        print(f'  TRM: failed ({e})')
        bench_results['trm'] = {'total_reward': 0, 'error': str(e)}

    results[bench_id] = bench_results

# Summary
print(f'\\n{"="*60}')
print('Summary (mean reward)')
print('='*60)
algos = set()
for br in results.values():
    algos.update(br.keys())
for algo in sorted(algos):
    rewards = [results[b][algo].get('total_reward', 0) for b in results if algo in results[b]]
    if rewards:
        print(f'  {algo:20s}: {np.mean(rewards):+8.4f} +/- {np.std(rewards):.4f}')

with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\\nResults saved to {results_path}')
''')

EVAL_BENCHES = 'qsort,adpcm,blowfish,bzip2,dijkstra,sha'
print(f'Running benchmark on: {EVAL_BENCHES}')

!$PYTHON $BENCHMARK_SCRIPT $PROJECT_DIR $MODEL_PATH $RESULTS_PATH $EVAL_BENCHES 30 42

In [ ]:
# Cell 9: Visualize benchmark results
import json
import matplotlib.pyplot as plt
import numpy as np

with open(RESULTS_PATH) as f:
    results = json.load(f)

benchmarks = list(results.keys())
algos = sorted(set(a for br in results.values() for a in br.keys()))
colors = {'llvm_Oz': '#2196F3', 'random_100': '#FF9800', 'trm': '#4CAF50'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(benchmarks))
width = 0.8 / max(len(algos), 1)

for ax_idx, (key, ylabel, title) in enumerate([
    ('total_reward', 'Total Reward', 'Reward by Algorithm (Real LLVM)'),
    ('reduction_pct', 'Reduction %', 'Instruction Reduction (Real LLVM)'),
]):
    for i, algo in enumerate(algos):
        vals = []
        for b in benchmarks:
            v = results[b].get(algo, {}).get(key, 0)
            vals.append(v * 100 if key == 'reduction_pct' else v)
        axes[ax_idx].bar(x + i * width, vals, width, label=algo,
                         color=colors.get(algo, '#999'), alpha=0.85)
    axes[ax_idx].set(xlabel='Benchmark', ylabel=ylabel, title=title)
    axes[ax_idx].set_xticks(x + width * (len(algos) - 1) / 2)
    axes[ax_idx].set_xticklabels(benchmarks, rotation=45, ha='right')
    axes[ax_idx].legend()
    axes[ax_idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print table
print(f"\n{'Benchmark':<15s} {'Algorithm':<15s} {'Reward':>8s} {'Reduction':>10s} {'Final Inst':>12s}")
print('-' * 65)
for b in benchmarks:
    for algo in algos:
        r = results[b].get(algo, {})
        rew = r.get('total_reward', 0)
        red = r.get('reduction_pct', 0) * 100
        inst = r.get('final_inst', '?')
        print(f'{b:<15s} {algo:<15s} {rew:+8.4f} {red:9.1f}% {inst:>12}')

## Detailed Rollout — Watch TRM on Real LLVM

In [ ]:
# Cell 10: Detailed rollout on a single benchmark

DETAIL_SCRIPT = f"{PROJECT_DIR}/trm_compiler/detail_rollout.py"

with open(DETAIL_SCRIPT, 'w') as f:
    f.write('''
import sys, torch
sys.path.insert(0, sys.argv[1])

from trm_compiler.model import TinyPassOrderingRefiner, rollout_closed_loop
from trm_compiler.env_wrapper import NUM_PASSES, pass_id_to_name, CompilerGymWrapper

device = 'cpu'
model_path = sys.argv[2]
bench_id = sys.argv[3]
max_steps = int(sys.argv[4])

model = TinyPassOrderingRefiner(observation_dim=56, latent_dim=64, hidden_dim=128,
    num_passes=NUM_PASSES, n_recursions=6)
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.eval()

wrapper = CompilerGymWrapper(benchmark_id=f'cbench-v1/{bench_id}')
wrapper.open()

trace = rollout_closed_loop(model, wrapper, max_steps=max_steps, temperature=1.0, device=device)

print(f'{\"Step\":>4s} {\"Pass\":<28s} {\"Reward\":>8s} {\"Inst\":>8s}')
print('-' * 55)
for s in trace:
    if s['pass_id'] >= 0:
        print(f'{s[\"step\"]:4d} {pass_id_to_name(s[\"pass_id\"]):<28s} '
              f'{s.get(\"real_reward\", 0):+8.4f} {s.get(\"inst_count\", \"?\")}')

print(f'\\nTotal reward: {trace[-1].get(\"total_real_reward\", 0):.4f}')
wrapper.close()
''')

!$PYTHON $DETAIL_SCRIPT $PROJECT_DIR $MODEL_PATH qsort 30

In [ ]:
# Cell 11: Save all outputs
import shutil

output_dir = f'{PROJECT_DIR}/trm_compiler_output'
print('Output files:')
for p in sorted(os.listdir(output_dir)):
    full = os.path.join(output_dir, p)
    sz = os.path.getsize(full)
    if sz > 1e6:
        print(f'  {p:40s} {sz/1e6:.1f} MB')
    else:
        print(f'  {p:40s} {sz/1e3:.0f} KB')